# Jour 5 — Comparaison SAC vs Baselines

Ce notebook compare :
- **SAC entraîné** — checkpoint Jour 3/4
- **Proportionnel** — refroidissement proportionnel à l'erreur de température
- **Bang-bang** — tout ou rien au seuil T_safe_max
- **Hystérésis** — bang-bang avec zone morte (moins oscillant)
- **Aléatoire** — borne inférieure

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import EnvConfig, ThermalConfig, RewardConfig
from envs.battery_thermal_env import BatteryThermalEnv
from evaluate import load_agent, run_episode
from baselines.rule_based import (
    RandomController, BangBangController,
    ProportionalController, HysteresisController,
)
from compare import evaluate_agent, print_table

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
CHECKPOINT  = '../runs/day3_run/checkpoints/sac_final.pt'
N_EPISODES  = 10

env_cfg = EnvConfig(thermal=ThermalConfig(), reward=RewardConfig())
env     = BatteryThermalEnv(config=env_cfg)
tc      = env_cfg.thermal

# --- Charger SAC ---
sac = load_agent(
    CHECKPOINT,
    obs_dim    = env.observation_space.shape[0],
    action_dim = env.action_space.shape[0],
)

agents = [
    ('SAC (entraîné)',  sac),
    ('Proportionnel',   ProportionalController(tc)),
    ('Bang-bang',       BangBangController(tc)),
    ('Hystérésis',      HysteresisController(tc)),
    ('Aléatoire',       RandomController(seed=42)),
]

print('Évaluation en cours...')
rows = []
for name, agent in agents:
    m = evaluate_agent(agent, env, n_episodes=N_EPISODES)
    rows.append((name, m))
    print(f'  {name:<20} return={m["return_mean"]:8.1f}  pct_safe={m["pct_safe_mean"]:.1f}%')

print('\nTerminé.')

In [ ]:
# Tableau résumé
print_table(rows)

In [ ]:
# Barplots comparatifs
names    = [r[0] for r in rows]
metrics  = [r[1] for r in rows]
colors   = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Return
ax = axes[0]
returns = [m['return_mean'] for m in metrics]
errs    = [m['return_std']  for m in metrics]
bars = ax.bar(names, returns, yerr=errs, color=colors, capsize=5, alpha=0.85)
ax.set_title('Return moyen (± std)', fontsize=12)
ax.set_ylabel('Return')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, returns):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(abs(e) for e in errs)*0.05,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

# % safe
ax = axes[1]
pct = [m['pct_safe_mean'] for m in metrics]
bars = ax.bar(names, pct, color=colors, alpha=0.85)
ax.set_title('% Steps en zone sûre', fontsize=12)
ax.set_ylabel('% steps')
ax.set_ylim(0, 115)
ax.axhline(100, color='green', linewidth=0.8, linestyle='--')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, pct):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../runs/day3_run/checkpoints/comparison_metrics.png', dpi=120)
plt.show()

In [ ]:
# Trajectoires température — meilleur épisode par politique
fig, ax = plt.subplots(figsize=(14, 6))

for (name, m), color in zip(rows, colors):
    best = max(m['results'], key=lambda r: r['return'])
    ax.plot(best['T_hist'], label=name, color=color, linewidth=1.3, alpha=0.85)

ax.axhline(tc.T_safe_min, color='blue',   linestyle='--', alpha=0.5, label=f'T_safe_min ({tc.T_safe_min}°C)')
ax.axhline(tc.T_safe_max, color='orange', linestyle='--', alpha=0.5, label=f'T_safe_max ({tc.T_safe_max}°C)')
ax.axhline(tc.T_cutoff,   color='red',    linestyle=':',  alpha=0.5, label=f'T_cutoff ({tc.T_cutoff}°C)')

max_len = max(len(m['results'][0]['T_hist']) for _, m in rows)
ax.fill_between(range(max_len), tc.T_safe_min, tc.T_safe_max, alpha=0.07, color='green', label='Zone sûre')

ax.set_title('Trajectoire température — meilleur épisode par politique', fontsize=12)
ax.set_xlabel('Step')
ax.set_ylabel('Température (°C)')
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig('../runs/day3_run/checkpoints/comparison_trajectories.png', dpi=120)
plt.show()

In [ ]:
# Distribution des actions par politique
fig, axes = plt.subplots(1, len(rows), figsize=(15, 4), sharey=True)

for ax, (name, m), color in zip(axes, rows, colors):
    all_actions = []
    for ep in m['results']:
        all_actions.extend(ep['action_hist'])
    ax.hist(all_actions, bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Action (cooling)')
    mean_a = np.mean(all_actions)
    ax.axvline(mean_a, color='black', linestyle='--', linewidth=1.5, label=f'μ={mean_a:.2f}')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Fréquence')
fig.suptitle('Distribution des actions par politique', fontsize=11)
plt.tight_layout()
plt.show()